# B07 — Pycol: kompleksitas cohort mammografi

Analisis **data training** CBIS-DDSM, CMMD, INbreast, dan MIAS. Tidak melakukan training model, mengubah split, atau membaca prediksi test.

**Alur:** cache B04 `full_512` → ResNet50 ImageNet beku → global average pooling → Pycol 1.0.4 → audit tetangga pasien → subset seimbang berulang.

Hasil adalah kompleksitas cohort **pada representasi fitur ini**, bukan kesulitan mutlak dataset atau bukti penyebab keberhasilan/kegagalan KD. INbreast memakai BI-RADS 1/2 vs 5/6; MIAS abnormal-only. Target antar-dataset tidak identik.

**Cara menjalankan:** buka runtime Colab baru (GPU L4/T4 dapat dipakai), jalankan Cell 1–9 berurutan. Default keempat dataset sekaligus. B04/B05/B06 tidak perlu dijalankan ulang. Folder Drive dari eksperimen sebelumnya harus tersedia. Cell 2 berisi lokasi registry; ubah hanya jika Drive Anda berbeda.

**Tiga keluaran:** (1) Pycol pada seluruh citra train, (2) diagnostik N2/N3 dengan tetangga dari pasien lain, (3) sensitivitas 20 subset, satu citra per pasien dengan jumlah negatif/positif yang sama untuk semua dataset. SD subset bukan confidence interval dan bukan variasi seed training.

Referensi: Apóstolo et al., *Pycol: A Python package for dataset complexity measures*, Neurocomputing 640 (2025), 130311. https://doi.org/10.1016/j.neucom.2025.130311

Paket: https://pypi.org/project/pycol-complexity/1.0.4/ ; repositori: https://github.com/DiogoApostolo/pycol

## Cell 1 — Lingkungan dan Google Drive
Paket Pycol dikunci pada 1.0.4. `tf-keras` disesuaikan dengan versi TensorFlow runtime. Gunakan runtime baru agar tidak mengubah sesi training aktif.

In [ ]:
import os, sys, subprocess, importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
subprocess.check_call([sys.executable,'-m','pip','install','-q','pycol-complexity==1.0.4'])
import tensorflow as tf
minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=minor:
    if 'tf_keras' in sys.modules: raise RuntimeError('Restart runtime sebelum mengubah tf-keras')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={minor}.0'])
import tf_keras as keras
import numpy as np, pandas as pd
import json, hashlib, zipfile, gc, time, inspect, platform
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import matplotlib.pyplot as plt
from pycol_complexity import complexity as pycol_module
from google.colab import drive, files
from IPython.display import display
gpus=tf.config.list_physical_devices('GPU')
for gpu in gpus: tf.config.experimental.set_memory_growth(gpu,True)
keras.mixed_precision.set_global_policy('float32')
keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
drive.mount('/content/drive')
print('TensorFlow:',tf.__version__,'tf-keras:',keras.__version__,'GPU:',gpus)
if not gpus: print('CPU tersedia, tetapi ekstraksi fitur akan lebih lambat.')


## Cell 2 — Konfigurasi
Default memakai keempat dataset agar ukuran subset pembanding dapat disamakan. Batch hanya memengaruhi ekstraksi fitur; tidak ada epoch atau early stopping.

In [ ]:
DATASETS=['cbis_ddsm','cmmd','inbreast','mias']
BATCH_SIZE=8  # turunkan ke 4 bila memori GPU tidak cukup
REPEATS=20
SUBSET_SEED=42  # seed pemilihan subset, bukan training
SUBSET_CAP_PER_CLASS=250
MAX_DISTANCE_MEMORY_GB=3.0
DOWNLOAD_REPORT=True
RESEARCH=Path('/content/drive/MyDrive/Research')
REGISTRY_PATH=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
SOURCE_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1'
SOURCE_IDS={'cbis_ddsm':'7a4bf6d1e3f6','cmmd':'f81ede1d807a','inbreast':'3f2fbdc2b913','mias':'672f572d6c33'}
OUTPUT_ROOT=RESEARCH/'outputs/benchmark_4datasets/b07_pycol_v1'
EXPECTED_PYCOL_SHA='6214fd699bfeeeade69bdebd7c99b0b5ba88e5d426b466e421f4f89846f0c4d6'
assert len(set(DATASETS))==len(DATASETS) and set(DATASETS)<=set(SOURCE_IDS)
assert DATASETS and REPEATS>=2 and BATCH_SIZE>=1
if set(DATASETS)!=set(SOURCE_IDS): print('PERHATIAN: subset disamakan hanya untuk dataset yang dipilih.')


## Cell 3 — Mesin analisis dan pemeriksaan implementasi
Pycol 1.0.4 menghitung jarak numerik HEOM dengan loop Python. Untuk efisiensi, notebook mengganti **hanya perhitungan matriks jarak** dengan operasi vektor yang ekuivalen. N1/N2/N3/F1 tetap memakai fungsi Pycol. Self-test membandingkan hasil dengan implementasi asli.

HEOM membagi setiap fitur dengan rentangnya pada cohort/subset yang sedang dihitung. Artinya, tiap subset memiliki normalisasi rentangnya sendiri sesuai perilaku paket. Tidak ada PCA, seleksi fitur berdasarkan hasil test, atau checkpoint Teacher terlatih yang dipakai.

N1 dengan jarak nol/sangat kecil dan N2 dengan jarak lintas kelas nol ditandai tidak tersedia karena berisiko menghasilkan interpretasi salah. Fisher dengan denominator degenerat juga ditandai; tidak diubah menjadi angka nol.

In [ ]:
ENGINE_SOURCE='import hashlib, json, os, gc, warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial.distance import cdist\nfrom pycol_complexity.complexity import Complexity\n\ndef file_sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef object_sha(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False).encode()).hexdigest()\n\ndef save_json(path,obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)); os.replace(tmp,path)\n\ndef read_json(path): return json.loads(Path(path).read_text())\n\nclass FastNumericComplexity(Complexity):\n    """Pycol 1.0.4, identical numeric HEOM via vectorized scipy cdist.\n    No categorical/missing input. Metric methods remain upstream Pycol.\n    Per-cohort range normalization is upstream behavior, not a shared scaler.\n    """\n    def _Complexity__calculate_distance_matrix(self,X,distance_func=\'default\'):\n        X=np.asarray(X,dtype=np.float64)\n        if distance_func not in (\'default\',\'HEOM\') or not np.isfinite(X).all():\n            raise ValueError(\'Only finite numeric HEOM input is supported\')\n        span=np.ptp(X,axis=0); span[span==0]=1.\n        Z=(X-X.min(axis=0))/span\n        return cdist(Z,Z),cdist(X,X)\n\ndef make_complexity(X,y):\n    X=np.asarray(X,dtype=np.float64); y=np.asarray(y,dtype=int)\n    if X.ndim!=2 or len(X)!=len(y) or not np.isfinite(X).all(): raise ValueError(\'Invalid features\')\n    if set(y)!={0,1} or min(np.bincount(y))<2: raise ValueError(\'Need >=2 images in each class\')\n    return FastNumericComplexity(file_type=\'array\',dataset={\'X\':X,\'y\':y,\'meta\':[0]*X.shape[1]})\n\ndef self_test():\n    rng=np.random.default_rng(917)\n    X=rng.normal(size=(14,7)); X[:7]+=1.5; X[:,6]=2.\n    y=np.array([0]*7+[1]*7)\n    a=Complexity(file_type=\'array\',dataset={\'X\':X,\'y\':y,\'meta\':[0]*7})\n    b=make_complexity(X,y)\n    np.testing.assert_allclose(a.dist_matrix,b.dist_matrix,rtol=1e-12,atol=1e-12)\n    np.testing.assert_allclose(a.unnorm_dist_matrix,b.unnorm_dist_matrix,rtol=1e-12,atol=1e-12)\n    for method in (\'N1\',\'N2\',\'N3\'):\n        for imb in (False,True):\n            np.testing.assert_allclose(getattr(a,method)(imb=imb),getattr(b,method)(imb=imb),rtol=1e-12,atol=1e-12)\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\',RuntimeWarning)\n        np.testing.assert_allclose(a.F1(),b.F1(),rtol=1e-12,atol=1e-12)\n    return {\'numeric_HEOM_matches_upstream\':True,\'N1_N2_N3_F1_match\':True,\'test_seed\':917}\n\ndef measure(X,frame,dataset,scenario=\'full_train\',repeat=-1,patient_checks=True):\n    """Long-format metrics; extra patient-excluded quantities are NOT native Pycol."""\n    y=frame.label.to_numpy(dtype=int); groups=frame.patient_id.to_numpy(str)\n    c=make_complexity(X,y); rows=[]\n    def add(metric,scope,value,implementation=\'pycol_1.0.4\',status=\'ok\'):\n        value=float(value)\n        rows.append(dict(dataset=dataset,scenario=scenario,repeat=int(repeat),metric=metric,\n            scope=str(scope),value=value if np.isfinite(value) else np.nan,status=status,\n            implementation=implementation,images=len(y),patients=len(set(groups)),\n            negative=int((y==0).sum()),positive=int((y==1).sum())))\n    off=c.dist_matrix[np.triu_indices(len(y),1)]\n    cross=c.dist_matrix[y==0][:,y==1]\n    for metric in (\'N1\',\'N2\',\'N3\'):\n        bad=(metric==\'N1\' and (off<=1e-8).any()) or (metric==\'N2\' and (cross==0).any())\n        if bad:\n            reason=\'undefined: zero/near-zero MST edges\' if metric==\'N1\' else \'undefined: cross-class zero distance\'\n            for scope in (\'overall\',\'class_0\',\'class_1\',\'macro\'): add(metric,scope,np.nan,status=reason)\n            continue\n        overall=getattr(c,metric)(); per=np.asarray(getattr(c,metric)(imb=True))\n        add(metric,\'overall\',overall)\n        for label,value in zip(c.classes,per): add(metric,f\'class_{label}\',value)\n        add(metric,\'macro\',per.mean(),implementation=\'mean_of_pycol_class_values\')\n    within=X[y==0].var(axis=0)+X[y==1].var(axis=0)\n    different=X[y==0].mean(axis=0)!=X[y==1].mean(axis=0)\n    active=np.ptp(X,axis=0)>0\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\',RuntimeWarning)\n        f1=np.asarray(c.F1(),dtype=float)\n    ftable=pd.DataFrame({\'feature_index\':np.arange(X.shape[1]),\'pycol_F1_overlap\':f1,\n        \'nonconstant\':active,\'zero_within_variance_different_means\':(within==0)&different})\n    good=active & ~((within==0)&different)\n    # Never silently trust upstream F1\'s inf->0 conversion on perfect constant-per-class features.\n    invalid=bool(((within==0)&different).any()) or not active.any()\n    for metric,fun in [(\'Fisher_overlap_min\',np.min),(\'Fisher_overlap_mean\',np.mean)]:\n        add(metric,\'overall\',np.nan if invalid else fun(f1[good]),\n            implementation=\'summary_of_pycol_F1_nonconstant_features\',\n            status=\'undefined: degenerate Fisher denominator\' if invalid else \'ok\')\n    D=c.dist_matrix.copy(); np.fill_diagonal(D,np.inf)\n    nearest=np.argmin(D,axis=1); nd=D[np.arange(len(y)),nearest]\n    ties=np.sum(np.isclose(D,nd[:,None],rtol=1e-12,atol=1e-12),axis=1)>1\n    detail=frame[[\'sample_id\',\'patient_id\',\'label\']].copy()\n    detail[\'nearest_sample_id\']=frame.sample_id.to_numpy()[nearest]\n    detail[\'nearest_same_patient\']=groups==groups[nearest]\n    detail[\'nearest_distance_tie\']=ties\n    if patient_checks:\n        D[groups[:,None]==groups[None,:]]=np.inf\n        ix=np.argmin(D,axis=1); finite=np.isfinite(D[np.arange(len(y)),ix])\n        error=(y!=y[ix]).astype(float); error[~finite]=np.nan\n        same=y[:,None]==y[None,:]\n        within_dist=np.min(np.where(same,D,np.inf),axis=1)\n        between_dist=np.min(np.where(~same,D,np.inf),axis=1)\n        for scope,mask in [(\'overall\',np.ones(len(y),bool)),(\'class_0\',y==0),(\'class_1\',y==1)]:\n            ok=finite[mask].all()\n            add(\'N3_other_patient\',scope,np.mean(error[mask]) if ok else np.nan,\n                \'custom_patient_excluded_1NN\',\'ok\' if ok else \'undefined: no eligible neighbour\')\n            a=within_dist[mask]; b=between_dist[mask]\n            ok=np.isfinite(a).all() and np.isfinite(b).all() and b.sum()>0\n            add(\'N2_other_patient\',scope,a.sum()/(a.sum()+b.sum()) if ok else np.nan,\n                \'custom_patient_excluded_distance_ratio\',\'ok\' if ok else \'undefined: missing eligible class neighbour\')\n        for metric in (\'N2_other_patient\',\'N3_other_patient\'):\n            vals=[r[\'value\'] for r in rows if r[\'metric\']==metric and r[\'scope\'] in (\'class_0\',\'class_1\')]\n            add(metric,\'macro\',np.mean(vals),\'mean_of_custom_class_values\',\n                \'ok\' if np.isfinite(vals).all() else \'undefined: class estimate unavailable\')\n        detail[\'other_patient_nearest_sample_id\']=frame.sample_id.to_numpy()[ix]\n        detail[\'other_patient_1NN_error\']=error\n    audit=dict(images=len(y),features=X.shape[1],constant_features=int((~active).sum()),\n        zero_distance_pairs=int((off==0).sum()),near_zero_pairs_MST=int((off<=1e-8).sum()),\n        nearest_tie_images=int(ties.sum()),nearest_same_patient_fraction=float(detail.nearest_same_patient.mean()),\n        mixed_label_patients=int((frame.groupby(\'patient_id\').label.nunique()>1).sum()))\n    del c,D; gc.collect()\n    return pd.DataFrame(rows),ftable,detail,audit\n\ndef balanced_patient_plans(frames,repeats=20,seed=42,cap_per_class=250):\n    """One random image/patient, then equal class counts shared across datasets/repeats.\n    Mixed-label patients may represent a different label in different repeats.\n    This changes class prevalence and cohort: sensitivity only, not a CI.\n    """\n    candidates={}; counts=[]\n    for ds,f in frames.items():\n        for repeat in range(repeats):\n            rng=np.random.default_rng(np.random.SeedSequence([seed,repeat,int(hashlib.sha256(ds.encode()).hexdigest()[:8],16)]))\n            ix=np.array([rng.choice(np.asarray(g,dtype=int)) for g in f.groupby(\'patient_id\',sort=True).indices.values()])\n            candidates[(ds,repeat)]=ix\n            counts.extend([(f.iloc[ix].label==label).sum() for label in (0,1)])\n    k=int(min(cap_per_class,min(counts)))\n    if k<2: raise RuntimeError(\'Not enough distinct patients per class for balanced sensitivity\')\n    plans={}\n    for (ds,repeat),ix in candidates.items():\n        rng=np.random.default_rng(seed+10000+repeat)\n        chosen=np.concatenate([rng.choice(ix[frames[ds].iloc[ix].label.to_numpy()==label],k,replace=False) for label in (0,1)])\n        plans[(ds,repeat)]=np.sort(chosen)\n        assert frames[ds].iloc[chosen].patient_id.is_unique\n    return k,plans\n'
engine_path=Path('/content/b07_pycol_engine.py')
engine_path.write_text(ENGINE_SOURCE)
exec(compile(ENGINE_SOURCE,str(engine_path),'exec'),globals())
pycol_source=Path(inspect.getfile(pycol_module))
if metadata.version('pycol-complexity')!='1.0.4' or file_sha(pycol_source)!=EXPECTED_PYCOL_SHA:
    raise RuntimeError('Versi/source Pycol berubah. Jangan lanjut sebelum audit implementasi.')
SELF_TEST=self_test()
print(SELF_TEST)


## Cell 4 — Audit cohort dan baca citra train
Memverifikasi hash CSV dan arsip terhadap READY.json serta protokol B05B. CSV validation dibaca **hanya untuk pemeriksaan identitas dan leakage**; fitur hanya dihitung dari train. CSV/citra test tidak dibuka. Hash arsip mencakup seluruh arsip secara opaque.

Tidak menggabungkan label pasien yang berbeda. Citra dari pasien dengan dua label tetap memiliki label citranya sendiri. Tidak ada citra hilang/rusak yang dilewati diam-diam.

In [ ]:
registry=read_json(REGISTRY_PATH)
FRAMES={}; IMAGE_DIRS={}; DATA_META={}; PROVENANCE={}; audit_rows=[]
for ds in DATASETS:
    print('Audit:',ds,flush=True)
    entry=registry[ds]['full_512']
    if entry['status']!='READY': raise RuntimeError(f'{ds}: cache belum READY')
    folder=Path(entry['path']); meta=read_json(folder/'READY.json')
    ppath=SOURCE_ROOT/ds/SOURCE_IDS[ds]/'protocol.json'; proto=read_json(ppath)
    if meta['dataset']!=ds or meta['image_size']!=512: raise RuntimeError('Cache dataset/size salah')
    if meta['file_sha256']!=proto['data_sha256'] or meta['cohort_sha256']!=proto['cohort_sha256']:
        raise RuntimeError(f'{ds}: cohort/cache tidak sesuai B05B')
    frames={}
    for split in ['train','validation']:
        path=folder/(split+'.csv')
        if file_sha(path)!=meta['file_sha256'][path.name]: raise RuntimeError(f'CSV berubah: {path}')
        f=pd.read_csv(path,dtype={'sample_id':str,'patient_id':str,'cache_member':str})
        required=['sample_id','patient_id','label','split','cache_member','processed_pixel_sha256']
        if not set(required)<=set(f) or f[required].isna().any().any(): raise ValueError(f'{ds}: ID/label/cache missing')
        if f.sample_id.duplicated().any() or f.cache_member.duplicated().any(): raise ValueError('Duplicate sample/member')
        if not f.label.isin([0,1]).all() or set(f.label)!={0,1} or set(f.split)!={split}: raise ValueError('Invalid split/label')
        if f.patient_id.str.strip().eq('').any(): raise ValueError('Empty patient_id')
        frames[split]=f.sort_values('sample_id').reset_index(drop=True)
    a,b=frames['train'],frames['validation']
    for col in ['sample_id','patient_id','processed_pixel_sha256']:
        if set(a[col])&set(b[col]): raise RuntimeError(f'{ds}: overlap train-validation pada {col}')
    estimate=8*len(a)**2*8/(1024**3)
    if estimate>MAX_DISTANCE_MEMORY_GB: raise MemoryError(f'{ds}: estimated distance work arrays {estimate:.2f} GiB; review memory budget')
    archive_path=folder/'images.zip'
    if file_sha(archive_path)!=meta['file_sha256']['images.zip']: raise RuntimeError('Archive hash berubah')
    image_dir=Path('/content/b07_train_images')/ds/meta['file_sha256']['images.zip'][:16]
    image_dir.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for row in a.itertuples():
            member=row.cache_member
            if Path(member).name!=member or '\\' in member: raise ValueError('Unsafe archive member')
            dest=image_dir/member
            if not dest.exists(): dest.write_bytes(archive.read(member))
            with Image.open(dest) as im: pixels=np.asarray(im)
            if pixels.dtype!=np.uint8 or pixels.shape!=(512,512) or hashlib.sha256(pixels.tobytes()).hexdigest()!=row.processed_pixel_sha256:
                raise RuntimeError(f'{ds}: pixel hash/shape mismatch {member}')
    FRAMES[ds]=a; IMAGE_DIRS[ds]=image_dir; DATA_META[ds]=meta
    PROVENANCE[ds]=dict(ready=meta,source_protocol_sha256=file_sha(ppath),source_protocol_id=SOURCE_IDS[ds],
        registry_sha256=file_sha(REGISTRY_PATH),ordered_train_ids_sha256=object_sha(a.sample_id.tolist()))
    counts=a.label.value_counts()
    audit_rows.append(dict(dataset=ds,images=len(a),patients=a.patient_id.nunique(),negative=int(counts[0]),
        positive=int(counts[1]),positive_prevalence=float(a.label.mean()),
        mixed_label_patients=int((a.groupby('patient_id').label.nunique()>1).sum()),
        duplicate_pixel_rows=int(a.processed_pixel_sha256.duplicated().sum()),estimated_distance_GiB=estimate))
COHORT_AUDIT=pd.DataFrame(audit_rows)
display(COHORT_AUDIT)
print('Audit selesai. Fitur hanya dari train. Test tidak dibuka.')


## Cell 5 — ResNet50 ImageNet beku dan cache fitur
Model sama untuk semua dataset: `include_top=False`, `pooling='avg'`, input RGB 512×512 dari cache grayscale (kanal direplikasi), preprocessing ResNet50, tanpa augmentasi. Output 2.048 fitur/citra.

Tidak ada training/fine-tuning. Ini representasi ImageNet, bukan representasi mammografi optimal. Kesimpulan harus dibatasi pada representasi ini. Cache yang lengkap dan hash-nya cocok akan digunakan kembali; cache B04/B05/B06 tidak ditimpa.

In [ ]:
keras.backend.clear_session(); keras.utils.set_random_seed(42)
extractor=keras.applications.ResNet50(include_top=False,weights='imagenet',input_shape=(512,512,3),pooling='avg')
extractor.trainable=False
weight_hash=hashlib.sha256()
for w in extractor.weights: weight_hash.update(w.numpy().tobytes())
environment={k:metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','pycol-complexity','Pillow']}
environment.update(tensorflow=tf.__version__,tf_keras=keras.__version__,python=platform.python_version(),
    gpu=[tf.config.experimental.get_device_details(g).get('device_name',str(g)) for g in gpus])
PROTOCOL=dict(version='b07_pycol_v1',datasets=DATASETS,split='train',backbone='ResNet50',weights='ImageNet',
    weights_sha256=weight_hash.hexdigest(),image_size=512,pooling='GAP',features=2048,augmentation='none',
    image_input='B04 full_512 uint8 PNG, grayscale replicated to RGB',preprocess='tf_keras.applications.resnet50.preprocess_input',
    scaling='Pycol numeric HEOM: per-feature range within each evaluated cohort/subset',
    batch_size=BATCH_SIZE,repeats=REPEATS,subset_seed=SUBSET_SEED,cap_per_class=SUBSET_CAP_PER_CLASS,
    subset='one random image per patient, then equal class counts; without replacement',
    pycol_source_sha256=file_sha(pycol_source),engine_sha256=hashlib.sha256(ENGINE_SOURCE.encode()).hexdigest(),
    provenance=PROVENANCE,environment=environment)
EXPERIMENT_ID=object_sha(PROTOCOL)[:16]
OUT=OUTPUT_ROOT/EXPERIMENT_ID; OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'protocol.json').exists() and read_json(OUT/'protocol.json')!=PROTOCOL: raise RuntimeError('Protocol conflict')
save_json(OUT/'protocol.json',PROTOCOL);save_json(OUT/'self_test.json',SELF_TEST)
COHORT_AUDIT.to_csv(OUT/'cohort_audit.csv',index=False)
(OUT/'b07_pycol_engine.py').write_text(ENGINE_SOURCE)
FEATURES={}
for ds in DATASETS:
    f=FRAMES[ds]; cache=OUT/'feature_cache'/ds; cache.mkdir(parents=True,exist_ok=True)
    manifest=dict(protocol_id=EXPERIMENT_ID,ordered_ids=f.sample_id.tolist(),weights_sha256=PROTOCOL['weights_sha256'])
    complete=cache/'COMPLETE.json'
    if complete.exists():
        done=read_json(complete)
        if done['manifest']!=manifest or file_sha(cache/'features.npy')!=done['features_sha256']:
            raise RuntimeError('Feature cache mismatch')
        X=np.load(cache/'features.npy')
        print(ds,'cache fitur digunakan kembali',flush=True)
    else:
        start=time.perf_counter(); chunks=[]
        for start_ix in range(0,len(f),BATCH_SIZE):
            members=f.cache_member.iloc[start_ix:start_ix+BATCH_SIZE]
            images=[]
            for member in members:
                with Image.open(IMAGE_DIRS[ds]/member) as im: images.append(np.asarray(im.convert('RGB'),dtype=np.float32))
            batch=np.stack(images)
            x=keras.applications.resnet50.preprocess_input(batch)
            z=extractor(x,training=False).numpy()
            if not np.isfinite(z).all(): raise ValueError('Nonfinite features')
            chunks.append(z.astype(np.float32))
            if start_ix%(BATCH_SIZE*25)==0: print(ds,start_ix,'/',len(f),flush=True)
        X=np.concatenate(chunks)
        tmp=cache/'features.tmp.npy';np.save(tmp,X);os.replace(tmp,cache/'features.npy')
        f[['sample_id','patient_id','label']].to_csv(cache/'feature_index.csv',index=False)
        save_json(complete,dict(manifest=manifest,features_sha256=file_sha(cache/'features.npy'),
            shape=list(X.shape),extraction_seconds=time.perf_counter()-start))
    if X.shape!=(len(f),2048) or not np.isfinite(X).all(): raise ValueError('Invalid cached feature shape/values')
    FEATURES[ds]=X
after=hashlib.sha256()
for w in extractor.weights: after.update(w.numpy().tobytes())
assert after.hexdigest()==PROTOCOL['weights_sha256'],'Extractor weights changed'
del extractor;keras.backend.clear_session();gc.collect()
print('Fitur siap. Output:',OUT)


## Cell 6 — Kompleksitas seluruh train dan audit pasien
`N1`, `N2`, `N3` adalah ukuran Pycol asli. Versi `macro` adalah rata-rata dua nilai per kelas, bukan skor global baru dari paper.

`N3_other_patient` dan `N2_other_patient` adalah **diagnostik tambahan buatan notebook**, bukan metrik Pycol resmi. Tetangga dari pasien yang sama dikeluarkan. N3 menggunakan satu tetangga; jika ada jarak sama, pilihan mengikuti urutan sample_id dan jumlah ties dicatat.

`Fisher_overlap_min` dan `Fisher_overlap_mean` merangkum keluaran `F1()` Pycol pada fitur nonkonstan. Makin tinggi berarti pemisahan lebih sulit menurut ukuran tersebut. Ini bukan F1-score klasifikasi. Raw per-fitur turut disimpan. Tidak ada rata-rata gabungan lintas jenis metrik atau ranking kesulitan tunggal.

In [ ]:
all_metrics=[]; distance_audits=[]
for ds in DATASETS:
    print('Pycol full train:',ds,flush=True)
    folder=OUT/'full_train'/ds;folder.mkdir(parents=True,exist_ok=True)
    table,fisher,neighbours,audit=measure(FEATURES[ds],FRAMES[ds],ds)
    table.to_csv(folder/'metrics.csv',index=False)
    fisher.to_csv(folder/'fisher_per_feature.csv',index=False)
    neighbours.to_csv(folder/'neighbour_audit.csv',index=False)
    save_json(folder/'distance_audit.json',audit)
    all_metrics.append(table);distance_audits.append(dict(dataset=ds,**audit))
FULL_METRICS=pd.concat(all_metrics,ignore_index=True)
FULL_METRICS.to_csv(OUT/'full_train_metrics.csv',index=False)
pd.DataFrame(distance_audits).to_csv(OUT/'distance_audit.csv',index=False)
display(FULL_METRICS[FULL_METRICS.scope=='overall'].pivot(index='dataset',columns='metric',values='value').round(4))
bad=FULL_METRICS[FULL_METRICS.status!='ok']
if len(bad):
    print('Ukuran tidak tersedia; baca alasan berikut. Tidak diganti dengan nol:')
    display(bad[['dataset','metric','scope','status']])


## Cell 7 — Sensitivitas ukuran cohort dan ketergantungan pasien
Untuk tiap pengulangan, pilih satu citra acak per pasien, kemudian ambil K citra negatif dan K citra positif tanpa penggantian. K sama untuk seluruh dataset dan pengulangan, dibatasi cohort terkecil dan cap pada Cell 2. Pasien dengan label campuran tetap hanya menyumbang satu citra; label yang terpilih bisa berubah antarpengulangan.

Ini mengubah prevalensi dan komposisi cohort sehingga **pelengkap** hasil full train. `mean`, sample `SD`, min–max menunjukkan variasi pemilihan subset, bukan bootstrap CI. Tidak menggandakan observasi dengan replacement karena duplikasi mengubah ukuran nearest-neighbour/MST.

In [ ]:
K,PLANS=balanced_patient_plans(FRAMES,REPEATS,SUBSET_SEED,SUBSET_CAP_PER_CLASS)
print('Setiap subset:',K,'negatif +',K,'positif; satu citra per pasien')
subset_tables=[]; selections=[]
for (ds,repeat),ix in PLANS.items():
    f=FRAMES[ds].iloc[ix].reset_index(drop=True)
    t,_,_,_=measure(FEATURES[ds][ix],f,ds,scenario='balanced_one_image_per_patient',repeat=repeat,patient_checks=False)
    subset_tables.append(t)
    selections.append(f[['sample_id','patient_id','label']].assign(dataset=ds,repeat=repeat))
    if repeat%5==0: print('Subset',ds,repeat+1,'/',REPEATS,flush=True)
SUBSET_METRICS=pd.concat(subset_tables,ignore_index=True)
SUBSET_METRICS.to_csv(OUT/'subset_metrics_all_repeats.csv',index=False)
pd.concat(selections,ignore_index=True).to_csv(OUT/'subset_membership.csv',index=False)
SUBSET_SUMMARY=SUBSET_METRICS.groupby(['dataset','metric','scope'],as_index=False).agg(
    mean=('value','mean'),sample_sd=('value','std'),minimum=('value','min'),maximum=('value','max'),
    valid_repeats=('value','count'),requested_repeats=('value','size'))
SUBSET_SUMMARY['images_per_class']=K
SUBSET_SUMMARY['status']=np.where(SUBSET_SUMMARY.valid_repeats==REPEATS,'complete','incomplete: inspect invalid repeats')
# Do not present means from only the convenient subset of valid repeats.
SUBSET_SUMMARY.loc[SUBSET_SUMMARY.valid_repeats!=REPEATS,['mean','sample_sd','minimum','maximum']]=np.nan
SUBSET_SUMMARY.to_csv(OUT/'subset_summary.csv',index=False)
save_json(OUT/'subset_design.json',dict(images_per_class=K,images_total=2*K,repeats=REPEATS,seed=SUBSET_SEED,
    replacement=False,uncertainty='subset sensitivity only; not a confidence interval'))
display(SUBSET_SUMMARY[SUBSET_SUMMARY.scope=='overall'].round(4))


## Cell 8 — Grafik dan panduan membaca
Grafik pertama memakai semua citra train. Grafik kedua memakai subset seimbang dengan error bar ±1 sample SD antarsubset (bukan CI). Jangan menggabungkan kedua grafik menjadi satu ranking dataset.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for ax,metric in zip(axes,['N1','N2','N3']):
    t=FULL_METRICS[(FULL_METRICS.metric==metric)&(FULL_METRICS.scope=='overall')].set_index('dataset').reindex(DATASETS)
    ax.bar(t.index,t.value,color='#356D9B');ax.set_title(metric+' — full train');ax.set_ylim(0,1);ax.tick_params(axis='x',rotation=20)
    ax.set_ylabel('Complexity (0–1)')
fig.savefig(OUT/'pycol_full_train.png',dpi=180);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for ax,metric in zip(axes,['N1','N2','N3']):
    t=SUBSET_SUMMARY[(SUBSET_SUMMARY.metric==metric)&(SUBSET_SUMMARY.scope=='overall')].set_index('dataset').reindex(DATASETS)
    valid=t['mean'].notna() & t.sample_sd.notna()
    if valid.any(): ax.bar(t.index[valid],t.loc[valid,'mean'],yerr=t.loc[valid,'sample_sd'],capsize=4,color='#56865C')
    else: ax.text(.5,.5,'Unavailable: inspect status',ha='center',transform=ax.transAxes)
    ax.set_title(metric+' — balanced subsets');ax.set_ylim(0,1);ax.tick_params(axis='x',rotation=20)
    ax.set_ylabel('Mean ± SD (not CI)')
fig.savefig(OUT/'pycol_subset_sensitivity.png',dpi=180);plt.show();plt.close(fig)
README="""B07 Pycol: train-cohort complexity in frozen ResNet50 ImageNet GAP features.
Full-train results and balanced subset sensitivity answer different questions.
Native Pycol measures: N1, N2, N3 and per-feature F1 overlap.
Fisher_overlap_min/mean summarize nonconstant features; NOT classification F1-score.
N2/N3_other_patient are custom diagnostics, excluding neighbours from the same patient.
Larger values indicate more overlap/difficulty for the selected measures, not all possible Pycol metrics.
Macro = unweighted average of per-class measure values. Do not average different metric types.
HEOM uses within-cohort/subset feature ranges. Feature representation and scaling affect results.
Subset SD is selection variability, not a confidence interval and not multi-seed training stability.
INbreast target BI-RADS 1/2 vs 5/6; MIAS abnormal-only. Targets/cohorts are not interchangeable.
No causal conclusion about KD follows from complexity scores. Test is not used for tuning here.
Missing/undefined values are documented in status fields, never substituted with zero.
Reference: https://doi.org/10.1016/j.neucom.2025.130311
Package: https://pypi.org/project/pycol-complexity/1.0.4/
"""
(OUT/'README.txt').write_text(README)
print('Baca full_train_metrics.csv bersama distance_audit.csv dan subset_summary.csv.')


## Cell 9 — ZIP laporan
ZIP memuat hasil, audit, protokol dan kode analisis. Citra dan matriks fitur tidak dimasukkan agar ukuran kecil. Cache fitur tetap berada di Drive. Nama report menggunakan timestamp sehingga laporan lama tidak tertimpa.

In [ ]:
REPORT_ROOT=OUTPUT_ROOT/'reports';REPORT_ROOT.mkdir(parents=True,exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
REPORT=REPORT_ROOT/f'B07_Pycol_{EXPERIMENT_ID}_{stamp}_report.zip'
report_files=[p for p in OUT.rglob('*') if p.is_file() and 'feature_cache' not in p.relative_to(OUT).parts and p.name!='report_manifest.json']
save_json(OUT/'report_manifest.json',dict(protocol_id=EXPERIMENT_ID,datasets=DATASETS,
    created_utc=stamp,full_train_metric_rows=len(FULL_METRICS),subset_metric_rows=len(SUBSET_METRICS),
    all_metrics_finite=bool(np.isfinite(FULL_METRICS.value).all() and np.isfinite(SUBSET_METRICS.value).all()),
    undefined_full_rows=int((FULL_METRICS.status!='ok').sum()),
    undefined_subset_rows=int((SUBSET_METRICS.status!='ok').sum()),
    files={str(p.relative_to(OUT)):file_sha(p) for p in report_files}))
with zipfile.ZipFile(REPORT,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for p in report_files+[OUT/'report_manifest.json']:archive.write(p,arcname=str(p.relative_to(OUT)))
print('Selesai:',REPORT,'| MB:',round(REPORT.stat().st_size/1024**2,2))
if DOWNLOAD_REPORT: files.download(str(REPORT))
